# Final Analysis: RAG Retrieval System Performance

This notebook performs a comprehensive analysis of the retrieval system's remaining failures after applying improvements (hybrid retrieval + reranking). We evaluate both **short queries (117)** and **long queries (41)**, identifying Top-1 failures, regressions, score diagnostics, and categorical patterns.

---

## 1. Setup & Data Loading

In [1]:
import json
import pandas as pd
from collections import Counter, defaultdict

# Load all data files
with open('Comparison_Results/short_queries_baseline_vs_improved.json', 'r', encoding='utf-8') as f:
    short_comparison = json.load(f)

with open('Comparison_Results/long_queries_baseline_vs_improved.json', 'r', encoding='utf-8') as f:
    long_comparison = json.load(f)

with open('evaluation_results/retrieval_evaluation.json', 'r', encoding='utf-8') as f:
    short_eval = json.load(f)

with open('evaluation_results/retrieval_evaluation_long.json', 'r', encoding='utf-8') as f:
    long_eval = json.load(f)

with open('retrieval_improvement_outputs/short_queries_117_reranked_outputs.json', 'r', encoding='utf-8') as f:
    short_reranked = json.load(f)

with open('retrieval_improvement_outputs/long_queries_41_reranked_outputs.json', 'r', encoding='utf-8') as f:
    long_reranked = json.load(f)

with open('failure_analysis_results/short_queries_117_retrieval_outputs.json', 'r', encoding='utf-8') as f:
    short_retrieval = json.load(f)

with open('failure_analysis_results/long_queries_41_retrieval_outputs.json', 'r', encoding='utf-8') as f:
    long_retrieval = json.load(f)

print(f"Short queries: {len(short_comparison)} | Long queries: {len(long_comparison)}")
print(f"Short reranked entries: {len(short_reranked)} | Long reranked entries: {len(long_reranked)}")
print(f"Short retrieval entries: {len(short_retrieval)} | Long retrieval entries: {len(long_retrieval)}")
print(f"Baseline evaluation - Short: {short_eval['total_queries']} | Long: {long_eval['total_queries']}")

Short queries: 116 | Long queries: 41
Short reranked entries: 116 | Long reranked entries: 41
Short retrieval entries: 116 | Long retrieval entries: 41
Baseline evaluation - Short: 116 | Long: 41


## 2. Overall Performance Summary

Compare baseline vs improved metrics across both query sets.

In [2]:
def compute_baseline_metrics(eval_data):
    """Compute Top-1, Top-3, Top-5 accuracy from baseline evaluation results."""
    results = eval_data['results']
    total = len(results)
    top1 = sum(1 for r in results if r['top_1'])
    top3 = sum(1 for r in results if r['top_3'])
    top5 = sum(1 for r in results if r['top_5'])
    return {
        'total': total,
        'top1_count': top1, 'top1_pct': round(top1/total*100, 2),
        'top3_count': top3, 'top3_pct': round(top3/total*100, 2),
        'top5_count': top5, 'top5_pct': round(top5/total*100, 2)
    }

def compute_improved_metrics(comparison_data):
    """Compute improved Top-1, Top-3, Top-5 accuracy from comparison data."""
    total = len(comparison_data)
    top1 = sum(1 for q in comparison_data if q['improved_rank_value'] == 1)
    top3 = sum(1 for q in comparison_data if q['improved_rank_value'] <= 3)
    top5 = sum(1 for q in comparison_data if q['improved_rank_value'] <= 5)
    return {
        'total': total,
        'top1_count': top1, 'top1_pct': round(top1/total*100, 2),
        'top3_count': top3, 'top3_pct': round(top3/total*100, 2),
        'top5_count': top5, 'top5_pct': round(top5/total*100, 2)
    }

# Compute metrics
short_baseline = compute_baseline_metrics(short_eval)
short_improved = compute_improved_metrics(short_comparison)
long_baseline = compute_baseline_metrics(long_eval)
long_improved = compute_improved_metrics(long_comparison)

# Build summary DataFrame
summary_data = {
    'Metric': ['Top-1 Accuracy', 'Top-3 Accuracy', 'Top-5 Accuracy'],
    'Short Baseline': [f"{short_baseline['top1_count']}/{short_baseline['total']} ({short_baseline['top1_pct']}%)",
                       f"{short_baseline['top3_count']}/{short_baseline['total']} ({short_baseline['top3_pct']}%)",
                       f"{short_baseline['top5_count']}/{short_baseline['total']} ({short_baseline['top5_pct']}%)"],
    'Short Improved': [f"{short_improved['top1_count']}/{short_improved['total']} ({short_improved['top1_pct']}%)",
                       f"{short_improved['top3_count']}/{short_improved['total']} ({short_improved['top3_pct']}%)",
                       f"{short_improved['top5_count']}/{short_improved['total']} ({short_improved['top5_pct']}%)"],
    'Long Baseline': [f"{long_baseline['top1_count']}/{long_baseline['total']} ({long_baseline['top1_pct']}%)",
                      f"{long_baseline['top3_count']}/{long_baseline['total']} ({long_baseline['top3_pct']}%)",
                      f"{long_baseline['top5_count']}/{long_baseline['total']} ({long_baseline['top5_pct']}%)"],
    'Long Improved': [f"{long_improved['top1_count']}/{long_improved['total']} ({long_improved['top1_pct']}%)",
                      f"{long_improved['top3_count']}/{long_improved['total']} ({long_improved['top3_pct']}%)",
                      f"{long_improved['top5_count']}/{long_improved['total']} ({long_improved['top5_pct']}%)"]
}

summary_df = pd.DataFrame(summary_data)
print("="*80)
print("OVERALL PERFORMANCE SUMMARY: BASELINE vs IMPROVED")
print("="*80)
print(summary_df.to_string(index=False))

# Status distribution
print("\n" + "="*80)
print("STATUS DISTRIBUTION")
print("="*80)
for name, data in [('Short Queries', short_comparison), ('Long Queries', long_comparison)]:
    statuses = Counter(q['status'] for q in data)
    print(f"\n{name}:")
    for status, count in sorted(statuses.items(), key=lambda x: -x[1]):
        print(f"  {status}: {count} ({round(count/len(data)*100, 1)}%)")

OVERALL PERFORMANCE SUMMARY: BASELINE vs IMPROVED
        Metric   Short Baseline   Short Improved  Long Baseline  Long Improved
Top-1 Accuracy  91/116 (78.45%)  96/116 (82.76%)  36/41 (87.8%) 37/41 (90.24%)
Top-3 Accuracy 102/116 (87.93%) 103/116 (88.79%) 39/41 (95.12%) 40/41 (97.56%)
Top-5 Accuracy 104/116 (89.66%) 106/116 (91.38%) 39/41 (95.12%) 41/41 (100.0%)

STATUS DISTRIBUTION

Short Queries:
  Unchanged: 99 (85.3%)
  Improved: 12 (10.3%)
  Regressed: 5 (4.3%)

Long Queries:
  Unchanged: 36 (87.8%)
  Improved: 3 (7.3%)
  Regressed: 2 (4.9%)


---
## 3. Extract All Remaining Top-1 Failures (Improved System)

A Top-1 failure is any query where the improved system did NOT rank the expected document at position 1.

In [3]:
def extract_top1_failures(comparison_data, reranked_data, retrieval_data, label=""):
    """Extract all queries where improved_rank_value != 1 (Top-1 failures in the improved system)."""
    failures = []
    
    # Build lookup dictionaries for reranked and retrieval data
    reranked_lookup = {}
    for entry in reranked_data:
        key = entry.get('query_id') or entry.get('query')
        reranked_lookup[key] = entry
    
    retrieval_lookup = {}
    for entry in retrieval_data:
        retrieval_lookup[entry['query']] = entry
    
    for q in comparison_data:
        if q['improved_rank_value'] != 1:
            query_id = q['query_id']
            query = q['query']
            expected = q['expected_document']
            baseline_rank = q['baseline_rank_value']
            improved_rank = q['improved_rank_value']
            status = q['status']
            
            # Get reranked details for score analysis
            reranked_entry = reranked_lookup.get(query_id, {})
            reranked_results = reranked_entry.get('results', [])
            
            # Get what was ranked #1 in the improved system
            top1_doc = reranked_results[0]['document'] if reranked_results else 'N/A'
            top1_rerank_score = reranked_results[0].get('rerank_score', 'N/A') if reranked_results else 'N/A'
            top1_hybrid_score = reranked_results[0].get('hybrid_score', 'N/A') if reranked_results else 'N/A'
            
            # Find expected doc's scores in reranked results
            expected_scores = {}
            for r in reranked_results:
                if r['document'] == expected:
                    expected_scores = {
                        'rerank_score': r.get('rerank_score', 'N/A'),
                        'hybrid_score': r.get('hybrid_score', 'N/A'),
                        'semantic_score': r.get('semantic_score', 'N/A'),
                        'bm25_score': r.get('bm25_score', 'N/A'),
                        'rank_in_improved': r.get('rank', 'N/A')
                    }
                    break
            
            # Get baseline retrieval details
            retrieval_entry = retrieval_lookup.get(query, {})
            baseline_results = retrieval_entry.get('results', [])
            baseline_top1_doc = baseline_results[0]['document'] if baseline_results else 'N/A'
            baseline_top1_sim = baseline_results[0].get('similarity_score', 'N/A') if baseline_results else 'N/A'
            
            failures.append({
                'query_id': query_id,
                'query': query,
                'expected_document': expected,
                'baseline_rank': baseline_rank,
                'improved_rank': improved_rank,
                'status': status,
                'improved_top1_doc': top1_doc,
                'improved_top1_rerank_score': top1_rerank_score,
                'improved_top1_hybrid_score': top1_hybrid_score,
                'expected_doc_rerank_score': expected_scores.get('rerank_score', 'Not in top-5'),
                'expected_doc_hybrid_score': expected_scores.get('hybrid_score', 'Not in top-5'),
                'expected_doc_semantic_score': expected_scores.get('semantic_score', 'Not in top-5'),
                'expected_doc_bm25_score': expected_scores.get('bm25_score', 'Not in top-5'),
                'baseline_top1_doc': baseline_top1_doc,
                'baseline_top1_similarity': baseline_top1_sim
            })
    
    return failures

short_failures = extract_top1_failures(short_comparison, short_reranked, short_retrieval, "Short")
long_failures = extract_top1_failures(long_comparison, long_reranked, long_retrieval, "Long")

print(f"Total Top-1 Failures in Improved System:")
print(f"  Short queries: {len(short_failures)}/{len(short_comparison)} ({round(len(short_failures)/len(short_comparison)*100, 1)}%)")
print(f"  Long queries:  {len(long_failures)}/{len(long_comparison)} ({round(len(long_failures)/len(long_comparison)*100, 1)}%)")

Total Top-1 Failures in Improved System:
  Short queries: 20/116 (17.2%)
  Long queries:  4/41 (9.8%)


### 3.1 Short Query Top-1 Failures (Detailed)

In [4]:
print("="*100)
print("SHORT QUERY TOP-1 FAILURES (IMPROVED SYSTEM)")
print("="*100)

for i, f in enumerate(short_failures, 1):
    print(f"\n{'---'*34}")
    print(f"Failure #{i} | Query ID: {f['query_id']} | Status: {f['status']}")
    print(f"{'---'*34}")
    print(f"  Query:              {f['query']}")
    print(f"  Expected Document:  {f['expected_document']}")
    print(f"  Baseline Rank:      {f['baseline_rank']}")
    print(f"  Improved Rank:      {f['improved_rank']}")
    print(f"  Improved Top-1 Doc: {f['improved_top1_doc']}")
    print(f"  ---")
    print(f"  Top-1 Doc Rerank Score:     {f['improved_top1_rerank_score']}")
    print(f"  Top-1 Doc Hybrid Score:     {f['improved_top1_hybrid_score']}")
    print(f"  Expected Doc Rerank Score:  {f['expected_doc_rerank_score']}")
    print(f"  Expected Doc Hybrid Score:  {f['expected_doc_hybrid_score']}")
    print(f"  Expected Doc Semantic Score:{f['expected_doc_semantic_score']}")
    print(f"  Expected Doc BM25 Score:    {f['expected_doc_bm25_score']}")
    print(f"  ---")
    print(f"  Baseline Top-1 Doc:         {f['baseline_top1_doc']}")
    print(f"  Baseline Top-1 Similarity:  {f['baseline_top1_similarity']}")

SHORT QUERY TOP-1 FAILURES (IMPROVED SYSTEM)

------------------------------------------------------------------------------------------------------
Failure #1 | Query ID: 6 | Status: Unchanged
------------------------------------------------------------------------------------------------------
  Query:              Is Thanksgiving a day off at Clef?
  Expected Document:  Holiday List.md
  Baseline Rank:      999.0
  Improved Rank:      999.0
  Improved Top-1 Doc: Vacation and Sick Leave.md
  ---
  Top-1 Doc Rerank Score:     1.2061405181884766
  Top-1 Doc Hybrid Score:     0.9064822348213717
  Expected Doc Rerank Score:  Not in top-5
  Expected Doc Hybrid Score:  Not in top-5
  Expected Doc Semantic Score:Not in top-5
  Expected Doc BM25 Score:    Not in top-5
  ---
  Baseline Top-1 Doc:         Communication and Transparency.md
  Baseline Top-1 Similarity:  0.2069278061389923

------------------------------------------------------------------------------------------------------
Fail

### 3.2 Long Query Top-1 Failures (Detailed)

In [5]:
print("="*100)
print("LONG QUERY TOP-1 FAILURES (IMPROVED SYSTEM)")
print("="*100)

for i, f in enumerate(long_failures, 1):
    print(f"\n{'---'*34}")
    print(f"Failure #{i} | Query ID: {f['query_id']} | Status: {f['status']}")
    print(f"{'---'*34}")
    print(f"  Query:              {f['query']}")
    print(f"  Expected Document:  {f['expected_document']}")
    print(f"  Baseline Rank:      {f['baseline_rank']}")
    print(f"  Improved Rank:      {f['improved_rank']}")
    print(f"  Improved Top-1 Doc: {f['improved_top1_doc']}")
    print(f"  ---")
    print(f"  Top-1 Doc Rerank Score:     {f['improved_top1_rerank_score']}")
    print(f"  Top-1 Doc Hybrid Score:     {f['improved_top1_hybrid_score']}")
    print(f"  Expected Doc Rerank Score:  {f['expected_doc_rerank_score']}")
    print(f"  Expected Doc Hybrid Score:  {f['expected_doc_hybrid_score']}")
    print(f"  Expected Doc Semantic Score:{f['expected_doc_semantic_score']}")
    print(f"  Expected Doc BM25 Score:    {f['expected_doc_bm25_score']}")
    print(f"  ---")
    print(f"  Baseline Top-1 Doc:         {f['baseline_top1_doc']}")
    print(f"  Baseline Top-1 Similarity:  {f['baseline_top1_similarity']}")

LONG QUERY TOP-1 FAILURES (IMPROVED SYSTEM)

------------------------------------------------------------------------------------------------------
Failure #1 | Query ID: 2 | Status: Unchanged
------------------------------------------------------------------------------------------------------
  Query:              My spouse and I are expecting a baby in a few months and I want to understand the full maternity and paternity leave policy including how it interacts with California state disability programs
  Expected Document:  New Parent Leave.md
  Baseline Rank:      2.0
  Improved Rank:      2
  Improved Top-1 Doc: Other Protected Absences.md
  ---
  Top-1 Doc Rerank Score:     -2.4071507453918457
  Top-1 Doc Hybrid Score:     0.9680233873685535
  Expected Doc Rerank Score:  -3.6681876182556152
  Expected Doc Hybrid Score:  0.9696996142157088
  Expected Doc Semantic Score:0.7787675261497498
  Expected Doc BM25 Score:    32.64507187380958
  ---
  Baseline Top-1 Doc:         New Parent

---
## 4. Regression Analysis

Queries where the improved system performs **worse** than baseline.

In [6]:
def extract_regressions(comparison_data, reranked_data, retrieval_data, label):
    """Extract queries that regressed (improved rank > baseline rank)."""
    regressions = []
    
    reranked_lookup = {entry.get('query_id', entry.get('query')): entry for entry in reranked_data}
    retrieval_lookup = {entry['query']: entry for entry in retrieval_data}
    
    for q in comparison_data:
        if q['status'] == 'Regressed':
            query_id = q['query_id']
            reranked_entry = reranked_lookup.get(query_id, {})
            reranked_results = reranked_entry.get('results', [])
            retrieval_entry = retrieval_lookup.get(q['query'], {})
            baseline_results = retrieval_entry.get('results', [])
            
            # Get the top-1 doc in improved and its scores
            improved_top1 = reranked_results[0] if reranked_results else {}
            
            # Find expected doc in reranked results
            expected_in_reranked = None
            for r in reranked_results:
                if r['document'] == q['expected_document']:
                    expected_in_reranked = r
                    break
            
            # Find expected doc in baseline results
            expected_in_baseline = None
            for r in baseline_results:
                if r['document'] == q['expected_document']:
                    expected_in_baseline = r
                    break
            
            regressions.append({
                'query_id': query_id,
                'query': q['query'],
                'expected_document': q['expected_document'],
                'baseline_rank': q['baseline_rank_value'],
                'improved_rank': q['improved_rank_value'],
                'rank_change': q['rank_change'],
                'improved_top1_doc': improved_top1.get('document', 'N/A'),
                'improved_top1_rerank': improved_top1.get('rerank_score', 'N/A'),
                'expected_rerank': expected_in_reranked.get('rerank_score', 'N/A') if expected_in_reranked else 'Not in Top-5',
                'expected_hybrid': expected_in_reranked.get('hybrid_score', 'N/A') if expected_in_reranked else 'Not in Top-5',
                'expected_semantic': expected_in_reranked.get('semantic_score', 'N/A') if expected_in_reranked else 'Not in Top-5',
                'baseline_similarity': expected_in_baseline.get('similarity_score', 'N/A') if expected_in_baseline else 'Not found'
            })
    
    return regressions

short_regressions = extract_regressions(short_comparison, short_reranked, short_retrieval, "Short")
long_regressions = extract_regressions(long_comparison, long_reranked, long_retrieval, "Long")

print(f"Total Regressions:")
print(f"  Short queries: {len(short_regressions)}")
print(f"  Long queries:  {len(long_regressions)}")

Total Regressions:
  Short queries: 5
  Long queries:  2


In [7]:
print("="*100)
print("REGRESSION DETAILS - SHORT QUERIES")
print("="*100)

for i, r in enumerate(short_regressions, 1):
    print(f"\n{'---'*34}")
    print(f"Regression #{i} | Query ID: {r['query_id']} | Rank: {r['baseline_rank']} -> {r['improved_rank']} (change: {r['rank_change']})")
    print(f"{'---'*34}")
    print(f"  Query:               {r['query']}")
    print(f"  Expected Document:   {r['expected_document']}")
    print(f"  Now Top-1:           {r['improved_top1_doc']}")
    print(f"  Top-1 Rerank Score:  {r['improved_top1_rerank']}")
    print(f"  Expected Rerank:     {r['expected_rerank']}")
    print(f"  Expected Hybrid:     {r['expected_hybrid']}")
    print(f"  Expected Semantic:   {r['expected_semantic']}")
    print(f"  Baseline Similarity: {r['baseline_similarity']}")

print("\n" + "="*100)
print("REGRESSION DETAILS - LONG QUERIES")
print("="*100)

for i, r in enumerate(long_regressions, 1):
    print(f"\n{'---'*34}")
    print(f"Regression #{i} | Query ID: {r['query_id']} | Rank: {r['baseline_rank']} -> {r['improved_rank']} (change: {r['rank_change']})")
    print(f"{'---'*34}")
    print(f"  Query:               {r['query']}")
    print(f"  Expected Document:   {r['expected_document']}")
    print(f"  Now Top-1:           {r['improved_top1_doc']}")
    print(f"  Top-1 Rerank Score:  {r['improved_top1_rerank']}")
    print(f"  Expected Rerank:     {r['expected_rerank']}")
    print(f"  Expected Hybrid:     {r['expected_hybrid']}")
    print(f"  Expected Semantic:   {r['expected_semantic']}")
    print(f"  Baseline Similarity: {r['baseline_similarity']}")

REGRESSION DETAILS - SHORT QUERIES

------------------------------------------------------------------------------------------------------
Regression #1 | Query ID: 8 | Rank: 1.0 -> 2.0 (change: -1.0)
------------------------------------------------------------------------------------------------------
  Query:               Do fathers get parental leave too or just mothers?
  Expected Document:   New Parent Leave.md
  Now Top-1:           Other Protected Absences.md
  Top-1 Rerank Score:  -2.220733165740967
  Expected Rerank:     -3.095492362976074
  Expected Hybrid:     0.9034616907803025
  Expected Semantic:   0.6723229885101318
  Baseline Similarity: 0.18270501494407654

------------------------------------------------------------------------------------------------------
Regression #2 | Query ID: 10 | Rank: 5.0 -> 999.0 (change: -994.0)
------------------------------------------------------------------------------------------------------
  Query:               I need some time awa

---
## 5. Failure Categorization (Root Cause Analysis)

Categorize failures by type to understand systematic issues.

In [8]:
def categorize_failures(failures):
    """Categorize each failure into root cause buckets."""
    categories = {
        'not_in_top5': [],         # Expected doc not in top-5 at all (rank 999)
        'reranker_demotion': [],    # Was in top-5 but reranker pushed it down (regressed)
        'close_miss': [],          # Rank 2-3 (near misses)
        'moderate_miss': [],       # Rank 4-5
        'total_miss': [],          # Rank > 5 (not in top-5)
    }
    
    for f in failures:
        improved_rank = f['improved_rank']
        
        if improved_rank == 999:
            categories['not_in_top5'].append(f)
        elif f['status'] == 'Regressed':
            categories['reranker_demotion'].append(f)
        elif improved_rank <= 3:
            categories['close_miss'].append(f)
        elif improved_rank <= 5:
            categories['moderate_miss'].append(f)
        else:
            categories['total_miss'].append(f)
    
    return categories

short_categories = categorize_failures(short_failures)
long_categories = categorize_failures(long_failures)

print("="*80)
print("FAILURE CATEGORIZATION")
print("="*80)

for label, cats in [('SHORT QUERIES', short_categories), ('LONG QUERIES', long_categories)]:
    print(f"\n{label}:")
    print(f"  Not in Top-5 (rank=999):     {len(cats['not_in_top5'])}")
    print(f"  Reranker Demotion:           {len(cats['reranker_demotion'])}")
    print(f"  Close Miss (rank 2-3):       {len(cats['close_miss'])}")
    print(f"  Moderate Miss (rank 4-5):    {len(cats['moderate_miss'])}")
    print(f"  Total Miss (rank >5):        {len(cats['total_miss'])}")

FAILURE CATEGORIZATION

SHORT QUERIES:
  Not in Top-5 (rank=999):     10
  Reranker Demotion:           4
  Close Miss (rank 2-3):       4
  Moderate Miss (rank 4-5):    2
  Total Miss (rank >5):        0

LONG QUERIES:
  Not in Top-5 (rank=999):     0
  Reranker Demotion:           2
  Close Miss (rank 2-3):       2
  Moderate Miss (rank 4-5):    0
  Total Miss (rank >5):        0


In [9]:
# Detail the NOT IN TOP-5 failures (most severe)
print("="*100)
print("MOST SEVERE FAILURES: Expected Document NOT in Top-5")
print("="*100)

all_not_in_top5 = []
for label, cats in [('SHORT', short_categories), ('LONG', long_categories)]:
    for f in cats['not_in_top5']:
        f_copy = f.copy()
        f_copy['query_type'] = label
        all_not_in_top5.append(f_copy)

for i, f in enumerate(all_not_in_top5, 1):
    print(f"\n{'---'*34}")
    print(f"#{i} [{f['query_type']}] Query ID: {f['query_id']}")
    print(f"  Query:     {f['query']}")
    print(f"  Expected:  {f['expected_document']}")
    print(f"  Got Top-1: {f['improved_top1_doc']}")
    print(f"  Baseline:  Rank={f['baseline_rank']} | Top-1 Doc={f['baseline_top1_doc']}")

print(f"\nTotal 'Not in Top-5' failures: {len(all_not_in_top5)}")

MOST SEVERE FAILURES: Expected Document NOT in Top-5

------------------------------------------------------------------------------------------------------
#1 [SHORT] Query ID: 6
  Query:     Is Thanksgiving a day off at Clef?
  Expected:  Holiday List.md
  Got Top-1: Vacation and Sick Leave.md
  Baseline:  Rank=999.0 | Top-1 Doc=Communication and Transparency.md

------------------------------------------------------------------------------------------------------
#2 [SHORT] Query ID: 10
  Query:     I need some time away, what are my options?
  Expected:  Vacation and Sick Leave.md
  Got Top-1: Welcome to Clef.md
  Baseline:  Rank=5.0 | Top-1 Doc=Communication and Transparency.md

------------------------------------------------------------------------------------------------------
#3 [SHORT] Query ID: 76
  Query:     What are Clef's core values?
  Expected:  Clef Values.md
  Got Top-1: Continuing Education.md
  Baseline:  Rank=999.0 | Top-1 Doc=Continuing Education.md

------------

---
## 6. Document-Level Failure Analysis

Which expected documents are most frequently missed?

In [10]:
# Combine all failures
all_failures = []
for f in short_failures:
    f_copy = f.copy()
    f_copy['query_type'] = 'Short'
    all_failures.append(f_copy)
for f in long_failures:
    f_copy = f.copy()
    f_copy['query_type'] = 'Long'
    all_failures.append(f_copy)

# Count failures by expected document
doc_failure_counts = Counter(f['expected_document'] for f in all_failures)

print("="*80)
print("DOCUMENTS MOST FREQUENTLY MISSED (Top-1 Failures)")
print("="*80)
print(f"{'Document':<45} {'Failures':>10} {'Avg Improved Rank':>20}")
print("-"*80)

for doc, count in doc_failure_counts.most_common():
    ranks = [f['improved_rank'] for f in all_failures if f['expected_document'] == doc]
    # Filter out 999 for average rank calculation
    valid_ranks = [r for r in ranks if r != 999]
    avg_rank = round(sum(valid_ranks) / len(valid_ranks), 1) if valid_ranks else 'N/A (all >5)'
    print(f"  {doc:<43} {count:>10}   {str(avg_rank):>18}")

DOCUMENTS MOST FREQUENTLY MISSED (Top-1 Failures)
Document                                        Failures    Avg Improved Rank
--------------------------------------------------------------------------------
  Clef Values.md                                       4                  3.0
  Welcome to Clef.md                                   3                  3.0
  New Parent Leave.md                                  2                  2.0
  Vacation and Sick Leave.md                           2                  3.0
  Mission Statement.md                                 2         N/A (all >5)
  Budgeting.md                                         2         N/A (all >5)
  Working Remotely.md                                  2                  3.5
  Holiday List.md                                      1         N/A (all >5)
  Salary and Equity Compensation.md                    1                  3.0
  Handbook Introduction.md                             1                  2.0
  Product M

In [11]:
# Which documents are most commonly retrieved as Top-1 when the expected doc fails?
confusing_docs = Counter(f['improved_top1_doc'] for f in all_failures)

print("="*80)
print("DOCUMENTS MOST COMMONLY RETRIEVED AS TOP-1 DURING FAILURES")
print("(These 'confuser' documents are displacing the correct results)")
print("="*80)
print(f"{'Document':<45} {'Times as Incorrect Top-1':>25}")
print("-"*80)
for doc, count in confusing_docs.most_common():
    print(f"  {doc:<43} {count:>25}")

DOCUMENTS MOST COMMONLY RETRIEVED AS TOP-1 DURING FAILURES
(These 'confuser' documents are displacing the correct results)
Document                                       Times as Incorrect Top-1
--------------------------------------------------------------------------------
  Continuing Education.md                                             6
  Other Protected Absences.md                                         2
  Welcome to Clef.md                                                  2
  One on Ones.md                                                      2
  Referral Bonuses.md                                                 2
  Working Remotely.md                                                 2
  Vacation and Sick Leave.md                                          1
  Equal Opportunity Employment.md                                     1
  Complaint Policy.md                                                 1
  Policy Changes.md                                                   1
  At

---
## 7. Reranker Score Diagnostics

Analyze the reranker's behavior on failures to understand if it's helping or hurting.

In [12]:
def analyze_reranker_scores(reranked_data, comparison_data):
    """Analyze reranker score distributions for successful and failed queries."""
    comp_lookup = {q['query_id']: q for q in comparison_data}
    
    success_top1_scores = []
    failure_top1_scores = []
    success_score_gaps = []
    failure_score_gaps = []
    
    for entry in reranked_data:
        qid = entry.get('query_id')
        results = entry.get('results', [])
        if not results or qid not in comp_lookup:
            continue
        
        comp = comp_lookup[qid]
        is_success = comp['improved_rank_value'] == 1
        
        top1_score = results[0].get('rerank_score', None)
        if top1_score is not None:
            if is_success:
                success_top1_scores.append(top1_score)
            else:
                failure_top1_scores.append(top1_score)
        
        # Score gap between rank 1 and rank 2
        if len(results) >= 2:
            score1 = results[0].get('rerank_score', 0)
            score2 = results[1].get('rerank_score', 0)
            if score1 is not None and score2 is not None:
                gap = score1 - score2
                if is_success:
                    success_score_gaps.append(gap)
                else:
                    failure_score_gaps.append(gap)
    
    return {
        'success_top1_scores': success_top1_scores,
        'failure_top1_scores': failure_top1_scores,
        'success_score_gaps': success_score_gaps,
        'failure_score_gaps': failure_score_gaps
    }

short_score_analysis = analyze_reranker_scores(short_reranked, short_comparison)
long_score_analysis = analyze_reranker_scores(long_reranked, long_comparison)

print("="*80)
print("RERANKER SCORE DIAGNOSTICS")
print("="*80)

for label, analysis in [('SHORT QUERIES', short_score_analysis), ('LONG QUERIES', long_score_analysis)]:
    print(f"\n{label}:")
    
    if analysis['success_top1_scores']:
        s_scores = analysis['success_top1_scores']
        print(f"  Successful queries Top-1 rerank score:")
        print(f"    Mean: {sum(s_scores)/len(s_scores):.3f} | Min: {min(s_scores):.3f} | Max: {max(s_scores):.3f} | Count: {len(s_scores)}")
    
    if analysis['failure_top1_scores']:
        f_scores = analysis['failure_top1_scores']
        print(f"  Failed queries Top-1 rerank score:")
        print(f"    Mean: {sum(f_scores)/len(f_scores):.3f} | Min: {min(f_scores):.3f} | Max: {max(f_scores):.3f} | Count: {len(f_scores)}")
    
    if analysis['success_score_gaps']:
        s_gaps = analysis['success_score_gaps']
        print(f"  Score gap (rank1 - rank2) for successes:")
        print(f"    Mean: {sum(s_gaps)/len(s_gaps):.3f} | Min: {min(s_gaps):.3f} | Max: {max(s_gaps):.3f}")
    
    if analysis['failure_score_gaps']:
        f_gaps = analysis['failure_score_gaps']
        print(f"  Score gap (rank1 - rank2) for failures:")
        print(f"    Mean: {sum(f_gaps)/len(f_gaps):.3f} | Min: {min(f_gaps):.3f} | Max: {max(f_gaps):.3f}")

RERANKER SCORE DIAGNOSTICS

SHORT QUERIES:
  Successful queries Top-1 rerank score:
    Mean: 2.866 | Min: -10.599 | Max: 10.057 | Count: 96
  Failed queries Top-1 rerank score:
    Mean: 2.560 | Min: -6.736 | Max: 9.854 | Count: 20
  Score gap (rank1 - rank2) for successes:
    Mean: 5.097 | Min: 0.524 | Max: 15.929
  Score gap (rank1 - rank2) for failures:
    Mean: 1.560 | Min: 0.147 | Max: 3.322

LONG QUERIES:
  Successful queries Top-1 rerank score:
    Mean: 2.222 | Min: -2.692 | Max: 5.645 | Count: 37
  Failed queries Top-1 rerank score:
    Mean: -2.526 | Min: -7.050 | Max: 1.981 | Count: 4
  Score gap (rank1 - rank2) for successes:
    Mean: 4.366 | Min: 0.514 | Max: 11.440
  Score gap (rank1 - rank2) for failures:
    Mean: 2.287 | Min: 0.353 | Max: 4.533


---
## 8. Improvement Success Analysis

Queries that improved from baseline to the current system.

In [13]:
print("="*100)
print("IMPROVEMENT SUCCESSES")
print("="*100)

for label, data in [('SHORT QUERIES', short_comparison), ('LONG QUERIES', long_comparison)]:
    improved = [q for q in data if q['status'] == 'Improved']
    print(f"\n{label} - {len(improved)} improved queries:")
    print(f"{'ID':>4} {'Baseline':>10} {'Improved':>10} {'Change':>8}  {'Query':<60}")
    print("-"*100)
    for q in improved:
        query_display = q['query'][:57] + '...' if len(q['query']) > 60 else q['query']
        print(f"{q['query_id']:>4} {q['baseline_rank_value']:>10} {q['improved_rank_value']:>10} {q['rank_change']:>+8.0f}  {query_display:<60}")

IMPROVEMENT SUCCESSES

SHORT QUERIES - 12 improved queries:
  ID   Baseline   Improved   Change  Query                                                       
----------------------------------------------------------------------------------------------------
  15        2.0        1.0       +1  I've been here 5 years, what's the deal with the long break?
  18        2.0        1.0       +1  Does Clef pay for online courses and books?                 
  46        2.0        1.0       +1  I have a family emergency, what should I do?                
  70        2.0        1.0       +1  Can we have beer at a company celebration?                  
  71      999.0        1.0     +998  Can Clef fire me without a reason?                          
  87        3.0        2.0       +1  Who is the CEO of Clef?                                     
  96        2.0        1.0       +1  What happens on Fridays in terms of team updates?           
  99        2.0        1.0       +1  Who do employees r

---
## 9. Reranker Behavior on Regressions (Deep Dive)

For each regression, compare the full reranked result set to understand what the reranker preferred and why.

In [14]:
def deep_dive_regression(query_id, reranked_data, retrieval_data, comparison_data):
    """Compare baseline retrieval ranking vs improved reranked ranking for a regression."""
    comp = next((q for q in comparison_data if q['query_id'] == query_id), None)
    reranked = next((e for e in reranked_data if e.get('query_id') == query_id), None)
    retrieval = next((e for e in retrieval_data if e['query'] == comp['query']), None) if comp else None
    
    if not all([comp, reranked]):
        print(f"  Data not found for query_id={query_id}")
        return
    
    print(f"\nQuery: {comp['query']}")
    print(f"Expected: {comp['expected_document']}")
    print(f"Baseline rank: {comp['baseline_rank_value']} -> Improved rank: {comp['improved_rank_value']}")
    
    # Show baseline top-5
    if retrieval and retrieval.get('results'):
        print(f"\n  BASELINE RETRIEVAL (similarity-only):")
        for r in retrieval['results'][:5]:
            marker = " << EXPECTED" if r['document'] == comp['expected_document'] else ""
            print(f"    Rank {r['rank']}: {r['document']:<45} sim={r['similarity_score']:.4f}{marker}")
    
    # Show improved top-5
    if reranked and reranked.get('results'):
        print(f"\n  IMPROVED (hybrid + reranking):")
        for r in reranked['results'][:5]:
            marker = " << EXPECTED" if r['document'] == comp['expected_document'] else ""
            print(f"    Rank {r['rank']}: {r['document']:<45} hybrid={r.get('hybrid_score', 0):.4f}  rerank={r.get('rerank_score', 0):.4f}{marker}")

print("="*100)
print("DEEP DIVE: ALL REGRESSIONS")
print("="*100)

print("\n--- SHORT QUERY REGRESSIONS ---")
for reg in short_regressions:
    print(f"\n{'==='*34}")
    deep_dive_regression(reg['query_id'], short_reranked, short_retrieval, short_comparison)

print(f"\n\n--- LONG QUERY REGRESSIONS ---")
for reg in long_regressions:
    print(f"\n{'==='*34}")
    deep_dive_regression(reg['query_id'], long_reranked, long_retrieval, long_comparison)

DEEP DIVE: ALL REGRESSIONS

--- SHORT QUERY REGRESSIONS ---


Query: Do fathers get parental leave too or just mothers?
Expected: New Parent Leave.md
Baseline rank: 1.0 -> Improved rank: 2.0

  BASELINE RETRIEVAL (similarity-only):
    Rank 1: New Parent Leave.md                           sim=0.1827 << EXPECTED
    Rank 2: Effective Meetings.md                         sim=0.1231
    Rank 3: Other Protected Absences.md                   sim=0.1214
    Rank 4: Other Protected Absences.md                   sim=0.1115
    Rank 5: Communication and Transparency.md             sim=0.1096

  IMPROVED (hybrid + reranking):
    Rank 1: Other Protected Absences.md                   hybrid=0.6862  rerank=-2.2207
    Rank 2: New Parent Leave.md                           hybrid=0.9035  rerank=-3.0955 << EXPECTED
    Rank 3: Other Protected Absences.md                   hybrid=0.8719  rerank=-7.5285
    Rank 4: Vacation and Sick Leave.md                    hybrid=0.5127  rerank=-11.2078
    Rank 5: 

---
## 10. Score Gap Analysis: Expected Doc vs Winning Doc

For each failure, compute the score gap between the winning doc and the expected doc across all scoring signals.

In [15]:
def score_gap_table(failures, reranked_data):
    """Build a table comparing winning doc scores vs expected doc scores."""
    reranked_lookup = {e.get('query_id'): e for e in reranked_data}
    
    rows = []
    for f in failures:
        qid = f['query_id']
        entry = reranked_lookup.get(qid)
        if not entry or not entry.get('results'):
            continue
        
        results = entry['results']
        winner = results[0]
        expected = None
        for r in results:
            if r['document'] == f['expected_document']:
                expected = r
                break
        
        if expected:
            rows.append({
                'query_id': qid,
                'query': f['query'][:50] + '...' if len(f['query']) > 50 else f['query'],
                'winner_doc': winner['document'],
                'expected_doc': f['expected_document'],
                'D_rerank': round(winner.get('rerank_score', 0) - expected.get('rerank_score', 0), 3),
                'D_hybrid': round(winner.get('hybrid_score', 0) - expected.get('hybrid_score', 0), 4),
                'D_semantic': round(winner.get('semantic_score', 0) - expected.get('semantic_score', 0), 4),
                'D_bm25': round(winner.get('bm25_score', 0) - expected.get('bm25_score', 0), 2),
                'winner_rerank': round(winner.get('rerank_score', 0), 3),
                'expected_rerank': round(expected.get('rerank_score', 0), 3)
            })
        else:
            rows.append({
                'query_id': qid,
                'query': f['query'][:50] + '...' if len(f['query']) > 50 else f['query'],
                'winner_doc': winner['document'],
                'expected_doc': f['expected_document'],
                'D_rerank': 'N/A (not in top-5)',
                'D_hybrid': 'N/A',
                'D_semantic': 'N/A',
                'D_bm25': 'N/A',
                'winner_rerank': round(winner.get('rerank_score', 0), 3),
                'expected_rerank': 'Not in top-5'
            })
    
    return pd.DataFrame(rows)

print("="*100)
print("SCORE GAP ANALYSIS: SHORT QUERY FAILURES")
print("="*100)
short_gap_df = score_gap_table(short_failures, short_reranked)
if not short_gap_df.empty:
    print(short_gap_df.to_string(index=False))

print("\n" + "="*100)
print("SCORE GAP ANALYSIS: LONG QUERY FAILURES")
print("="*100)
long_gap_df = score_gap_table(long_failures, long_reranked)
if not long_gap_df.empty:
    print(long_gap_df.to_string(index=False))

SCORE GAP ANALYSIS: SHORT QUERY FAILURES
 query_id                                                 query                      winner_doc                      expected_doc           D_rerank D_hybrid D_semantic D_bm25  winner_rerank expected_rerank
        6                    Is Thanksgiving a day off at Clef?      Vacation and Sick Leave.md                   Holiday List.md N/A (not in top-5)      N/A        N/A    N/A          1.206    Not in top-5
        8    Do fathers get parental leave too or just mothers?     Other Protected Absences.md               New Parent Leave.md              0.875  -0.2173    -0.0703  -1.53         -2.221          -3.095
       10           I need some time away, what are my options?              Welcome to Clef.md        Vacation and Sick Leave.md N/A (not in top-5)      N/A        N/A    N/A         -6.736    Not in top-5
       36                        How much do the founders make?         Continuing Education.md Salary and Equity Compensation.md  

---
## 11. Cross-System Comparison Summary Table

A combined view of every failure showing baseline rank, improved rank, and the nature of the problem.

In [16]:
def build_failure_summary_table(failures, label):
    """Build a summary table of all failures."""
    rows = []
    for f in failures:
        # Classify the failure type
        if f['improved_rank'] == 999:
            failure_type = 'NOT IN TOP-5'
        elif f['status'] == 'Regressed':
            failure_type = 'REGRESSION'
        elif f['baseline_rank'] == 999:
            failure_type = 'PARTIAL RECOVERY'
        else:
            failure_type = 'PERSISTENT MISS'
        
        rows.append({
            'Type': label,
            'QID': f['query_id'],
            'Query': f['query'][:55] + '...' if len(f['query']) > 55 else f['query'],
            'Expected': f['expected_document'],
            'Baseline': int(f['baseline_rank']) if f['baseline_rank'] != 999 else '>5',
            'Improved': int(f['improved_rank']) if f['improved_rank'] != 999 else '>5',
            'Got': f['improved_top1_doc'],
            'Category': failure_type
        })
    
    return pd.DataFrame(rows)

short_summary = build_failure_summary_table(short_failures, 'Short')
long_summary = build_failure_summary_table(long_failures, 'Long')
combined_summary = pd.concat([short_summary, long_summary], ignore_index=True)

print("="*120)
print("COMBINED FAILURE SUMMARY TABLE")
print("="*120)
print(combined_summary.to_string(index=False))

# Category counts
print("\n" + "-"*60)
print("FAILURE CATEGORY BREAKDOWN:")
print("-"*60)
for cat, count in combined_summary['Category'].value_counts().items():
    print(f"  {cat:<25} {count:>5}")

COMBINED FAILURE SUMMARY TABLE
 Type  QID                                                      Query                          Expected Baseline Improved                               Got         Category
Short    6                         Is Thanksgiving a day off at Clef?                   Holiday List.md       >5       >5        Vacation and Sick Leave.md     NOT IN TOP-5
Short    8         Do fathers get parental leave too or just mothers?               New Parent Leave.md        1        2       Other Protected Absences.md       REGRESSION
Short   10                I need some time away, what are my options?        Vacation and Sick Leave.md        5       >5                Welcome to Clef.md     NOT IN TOP-5
Short   36                             How much do the founders make? Salary and Equity Compensation.md        3        3           Continuing Education.md  PERSISTENT MISS
Short   76                               What are Clef's core values?                    Clef Values.md 

---
## 12. Synthesis & Recommendations

Final analysis and actionable recommendations based on all findings.

In [19]:
# Compute final statistics
total_short = len(short_comparison)
total_long = len(long_comparison)
total_queries = total_short + total_long

short_top1_success = total_short - len(short_failures)
long_top1_success = total_long - len(long_failures)

short_improved_count = sum(1 for q in short_comparison if q['status'] == 'Improved')
long_improved_count = sum(1 for q in long_comparison if q['status'] == 'Improved')
short_regressed_count = len(short_regressions)
long_regressed_count = len(long_regressions)

# Persistent failures (failed in both systems)
persistent_short = [f for f in short_failures if f['baseline_rank'] != 1]
persistent_long = [f for f in long_failures if f['baseline_rank'] != 1]

# NEW failures (was rank 1 in baseline, now not)
new_short_failures = [f for f in short_failures if f['baseline_rank'] == 1]
new_long_failures = [f for f in long_failures if f['baseline_rank'] == 1]

print("="*100)
print("FINAL SYNTHESIS & RECOMMENDATIONS")
print("="*100)

print("\n[OVERALL RESULTS]")
print(f"   Total queries evaluated: {total_queries} (Short: {total_short}, Long: {total_long})")
print(f"   Short Top-1 Accuracy: {short_top1_success}/{total_short} ({round(short_top1_success/total_short*100, 1)}%) <- Baseline: {short_baseline['top1_pct']}%")
print(f"   Long Top-1 Accuracy:  {long_top1_success}/{total_long} ({round(long_top1_success/total_long*100, 1)}%) <- Baseline: {long_baseline['top1_pct']}%")

print(f"\n[IMPROVEMENTS]")
print(f"   Short: {short_improved_count} queries improved")
print(f"   Long:  {long_improved_count} queries improved")

print(f"\n[REGRESSIONS]")
print(f"   Short: {short_regressed_count} queries regressed")
print(f"   Long:  {long_regressed_count} queries regressed")

print(f"\n[NEW FAILURES] (was Top-1 in baseline, now NOT Top-1):")
print(f"   Short: {len(new_short_failures)}")
for f in new_short_failures:
    print(f"     QID {f['query_id']}: {f['query'][:70]}")
print(f"   Long: {len(new_long_failures)}")
for f in new_long_failures:
    print(f"     QID {f['query_id']}: {f['query'][:70]}")

print(f"\n[PERSISTENT FAILURES] (failed in both baseline and improved):")
print(f"   Short: {len(persistent_short)}")
print(f"   Long:  {len(persistent_long)}")

print("\n" + "-"*100)
print("\nKEY FINDINGS:")
print("")
print("  1. RERANKER IMPACT: The reranker generally helps (more improvements than regressions),")
print("     but introduces regressions where it promotes semantically-related but incorrect")
print("     documents over the correct ones.")
print("")
print("  2. HARDEST DOCUMENTS: Documents like 'Clef Values.md', 'Mission Statement.md',")
print("     'Budgeting.md', and 'Effective Meetings.md' have queries that consistently")
print("     fail across both systems - these likely have chunking or content overlap issues.")
print("")
print("  3. SEMANTIC OVERLAP: Documents in the same category (e.g., 'Benefits and Perks')")
print("     share vocabulary, causing the retriever to confuse them.")
print("")

print("\nRECOMMENDATIONS:")
print("")
print("  1. RERANKER CONFIDENCE THRESHOLD: Apply a minimum confidence threshold to prevent")
print("     the reranker from reordering when score gaps are very small (<1.0 points).")
print("")
print("  2. CHUNK METADATA BOOST: Use document category/section metadata to boost scores")
print("     for chunks whose metadata closely matches query intent.")
print("")
print("  3. QUERY EXPANSION: For vague/general queries (e.g., 'What are Clef's core values?'),")
print("     consider query expansion to add more specific terms.")
print("")
print("  4. HYBRID WEIGHT TUNING: Review the semantic-to-BM25 weight ratio - some regressions")
print("     suggest BM25 keyword matching is being over-weighted for certain query types.")
print("")
print("  5. DOCUMENT-SPECIFIC CHUNKING: Re-examine chunking for consistently-failing documents")
print("     to ensure the key content is in a single, representative chunk.")

FINAL SYNTHESIS & RECOMMENDATIONS

[OVERALL RESULTS]
   Total queries evaluated: 157 (Short: 116, Long: 41)
   Short Top-1 Accuracy: 96/116 (82.8%) <- Baseline: 78.45%
   Long Top-1 Accuracy:  37/41 (90.2%) <- Baseline: 87.8%

[IMPROVEMENTS]
   Short: 12 queries improved
   Long:  3 queries improved

[REGRESSIONS]
   Short: 5 queries regressed
   Long:  2 queries regressed

[NEW FAILURES] (was Top-1 in baseline, now NOT Top-1):
   Short: 3
     QID 8: Do fathers get parental leave too or just mothers?
     QID 77: What does 'be better today than yesterday' mean?
     QID 82: What should I expect on my first day at Clef?
   Long: 1
     QID 22: I want to work from a coffee shop while traveling but I'm concerned ab

[PERSISTENT FAILURES] (failed in both baseline and improved):
   Short: 17
   Long:  3

----------------------------------------------------------------------------------------------------

KEY FINDINGS:

  1. RERANKER IMPACT: The reranker generally helps (more improvements t

In [18]:
# Final compact summary table
print("="*70)
print("COMPACT FINAL METRICS")
print("="*70)

metrics = [
    ['Short Queries (117)', 
     f"{short_baseline['top1_pct']}%", f"{short_improved['top1_pct']}%",
     f"{short_improved['top3_pct']}%", f"{short_improved['top5_pct']}%",
     str(short_improved_count), str(short_regressed_count)],
    ['Long Queries (41)', 
     f"{long_baseline['top1_pct']}%", f"{long_improved['top1_pct']}%",
     f"{long_improved['top3_pct']}%", f"{long_improved['top5_pct']}%",
     str(long_improved_count), str(long_regressed_count)],
]

print(f"{'Query Set':<22} {'BL Top-1':>10} {'IMP Top-1':>10} {'IMP Top-3':>10} {'IMP Top-5':>10} {'Improved':>10} {'Regressed':>10}")
print("-"*70)
for row in metrics:
    print(f"{row[0]:<22} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>10} {row[5]:>10} {row[6]:>10}")

print("\nAnalysis complete.")

COMPACT FINAL METRICS
Query Set                BL Top-1  IMP Top-1  IMP Top-3  IMP Top-5   Improved  Regressed
----------------------------------------------------------------------
Short Queries (117)        78.45%     82.76%     88.79%     91.38%         12          5
Long Queries (41)           87.8%     90.24%     97.56%     100.0%          3          2

Analysis complete.
